# Fine-Tune `Qwen2.5-3B-Instruct` on Free Colab T4 GPU (No GCP / No Billing)

This notebook fine-tunes an open-weight model (`Qwen2.5-3B-Instruct` in 4-bit LoRA) on your exported event dataset using **100% free Google Colab T4 GPU** — no Google Cloud project, no GCS bucket, and no credit card.

### Setup Checklist
1. In Colab menu: **Runtime -> Change runtime type -> T4 GPU -> Save**.
2. Have `gemini_sft_train.jsonl` (and optional `gemini_sft_val.jsonl`) ready from `E:\FITUS 26-27\HK4\TDTT\No-Le-DRL\events\tuning_data\`.

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install -q fastapi uvicorn

## 1. Upload `gemini_sft_train.jsonl` (from `events/tuning_data/`)
If you already dragged `gemini_sft_train.jsonl` into Colab's left **Files** panel, this cell detects it automatically without prompting again.

In [ ]:
import json
from pathlib import Path
from google.colab import files

if not Path('gemini_sft_train.jsonl').is_file():
    print('Select gemini_sft_train.jsonl (and optionally gemini_sft_val.jsonl):')
    files.upload()

def load_sft_jsonl(path_str: str):
    p = Path(path_str)
    if not p.is_file() or p.stat().st_size == 0:
        return []
    records = []
    for line in p.read_text(encoding='utf-8').splitlines():
        if not line.strip():
            continue
        obj = json.loads(line)
        sys_text = obj.get('systemInstruction', {}).get('parts', [{}])[0].get('text', '')
        contents = obj.get('contents', [])
        user_text = contents[0]['parts'][0]['text']
        model_text = contents[1]['parts'][0]['text']
        records.append({
            'messages': [
                {'role': 'system', 'content': sys_text},
                {'role': 'user', 'content': user_text},
                {'role': 'assistant', 'content': model_text},
            ]
        })
    return records

train_records = load_sft_jsonl('gemini_sft_train.jsonl')
val_records = load_sft_jsonl('gemini_sft_val.jsonl')
print(f'Loaded {len(train_records)} training examples and {len(val_records)} validation examples.')

## 2. Load 4-Bit `Qwen2.5-3B-Instruct` & Fine-Tune with LoRA (~10 mins on T4)

In [ ]:
import inspect
import torch
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import FastLanguageModel, is_bfloat16_supported

MAX_SEQ_LENGTH = 3072

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Qwen2.5-3B-Instruct-unsloth-bnb-4bit',
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)

def format_to_text(batch):
    texts = [
        tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
        for msgs in batch['messages']
    ]
    return {'text': texts}

train_ds = Dataset.from_list(train_records).map(format_to_text, batched=True)
eval_ds = Dataset.from_list(val_records).map(format_to_text, batched=True) if val_records else None

trainer_kwargs = dict(
    model=model,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=5,
        optim='adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='linear',
        seed=42,
        output_dir='outputs',
        report_to='none',
    ),
)
sig_params = inspect.signature(SFTTrainer.__init__).parameters
if 'tokenizer' in sig_params:
    trainer_kwargs['tokenizer'] = tokenizer
elif 'processing_class' in sig_params:
    trainer_kwargs['processing_class'] = tokenizer

trainer = SFTTrainer(**trainer_kwargs)
stats = trainer.train()
model.save_pretrained('lora_event_extractor')
tokenizer.save_pretrained('lora_event_extractor')
print('Training complete! Final loss:', stats.training_loss)

## 3. Quick Sanity Test Inside Colab

In [ ]:
FastLanguageModel.for_inference(model)
inference_sys = train_records[0]['messages'][0]['content'] if train_records else 'Extract university event JSON.'
sample_user = '''Post published at: 2026-11-24T09:00:00+07:00
Fanpage / Source: Đoàn - Hội Khoa CNTT HCMUS
Post text:
"""
[WORKSHOP: BUILD AGENTIC AI SYSTEMS 2026]
Thời gian: 14:00 - 17:00 ngày 30/11/2026
Địa điểm: Hội trường I, Trường ĐH KHTN ĐHQG-HCM (Cơ sở 1)
Lệ phí: Miễn phí
Đăng ký ngay tại: https://forms.gle/demo123 (Hạn chót: 28/11/2026)
"""'''

inputs = tokenizer.apply_chat_template(
    [{'role': 'system', 'content': inference_sys.strip()}, {'role': 'user', 'content': sample_user}],
    tokenize=True,
    add_generation_prompt=True,
    return_tensors='pt',
).to('cuda')
outputs = model.generate(input_ids=inputs, max_new_tokens=768, temperature=0.0, do_sample=False)
decoded = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
print(decoded)

## 4A. Serve Live From Colab via Free Cloudflare Tunnel (No Signup / No Billing)
Run this cell to expose an OpenAI-compatible `/v1/chat/completions` endpoint directly from Colab and copy `CUSTOM_LLM_URL=...` into your local `.env`.

In [ ]:
import re
import subprocess
import threading
import time
from pathlib import Path
from typing import List, Optional
import uvicorn
from fastapi import FastAPI
from pydantic import BaseModel

FastLanguageModel.for_inference(model)
gpu_lock = threading.Lock()
colab_app = FastAPI(title='Colab Event Extractor LLM')

class ChatMsg(BaseModel):
    role: str
    content: str

class ChatReq(BaseModel):
    model: Optional[str] = 'event-extractor'
    messages: List[ChatMsg]
    temperature: Optional[float] = 0.0
    max_tokens: Optional[int] = 1536

@colab_app.get('/health')
def health():
    return {'ok': True, 'model': 'Qwen2.5-3B-Instruct-LoRA'}

@colab_app.post('/v1/chat/completions')
def chat_completions(req: ChatReq):
    msgs = [{'role': m.role, 'content': m.content} for m in req.messages]
    with gpu_lock:
        input_ids = tokenizer.apply_chat_template(
            msgs,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors='pt',
        ).to('cuda')
        out_ids = model.generate(
            input_ids=input_ids,
            max_new_tokens=min(int(req.max_tokens or 1536), 2048),
            do_sample=False,
        )
        reply = tokenizer.decode(out_ids[0][input_ids.shape[-1]:], skip_special_tokens=True).strip()
    return {
        'id': 'chatcmpl-colab',
        'object': 'chat.completion',
        'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': reply}, 'finish_reason': 'stop'}],
    }

# Start FastAPI server in background thread
threading.Thread(
    target=lambda: uvicorn.run(colab_app, host='127.0.0.1', port=8000, log_level='warning'),
    daemon=True,
).start()
time.sleep(2)

# Download & launch free Cloudflare Quick Tunnel
if not Path('/usr/local/bin/cloudflared').is_file():
    subprocess.run(
        ['wget', '-q', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', '-O', '/usr/local/bin/cloudflared'],
        check=True,
    )
    subprocess.run(['chmod', '+x', '/usr/local/bin/cloudflared'], check=True)

proc = subprocess.Popen(
    ['/usr/local/bin/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

public_url = None
for line in proc.stdout:
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m:
        public_url = m.group(0)
        print('============================================================')
        print(' COLAB MODEL IS LIVE! Paste these 2 lines into your .env:')
        print(f'CUSTOM_LLM_URL={public_url}/v1/chat/completions')
        print('CUSTOM_LLM_MODEL=event-extractor')
        print('============================================================')
        break

# Keep cell running while serving requests
proc.wait()

## 4B. (Optional) Export Quantized `.gguf` to Run Locally 24/7 via Ollama
If you want to close Colab and run the model locally on your PC without tunnels, run this cell to export a 4-bit `.gguf` file and download it.

In [ ]:
model.save_pretrained_gguf('event_extractor_gguf', tokenizer, quantization_method='q4_k_m')
print('Saved GGUF inside /content/event_extractor_gguf/ — download the .gguf file from the left Files panel!')
print('Then locally create Modelfile with: FROM ./unsloth.Q4_K_M.gguf  ->  ollama create event-extractor -f Modelfile')
print('And set in .env: CUSTOM_LLM_URL=http://127.0.0.1:11434/v1/chat/completions')